In [4]:
import os
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
import numpy as np
import mlflow
import mlflow.pytorch
from model import SLRHybridModel


CONFIG = {
    "num_classes": 353,       
    "batch_size": 32,         
    "lr": 5e-4,  
    "epochs": 100,
    "patience": 20,       
    "input_dim": 99,
    "hidden_dim": 64,        
    "experiment_name": "SSL400-Hybrid-Paper-Replication",
     "mlflow_tracking_uri": "sqlite:///mlflow.db"
}

def train_hybrid():
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    
  
    data_dir = "../../ssl400_processed_paper"
    X_train = torch.from_numpy(np.load(os.path.join(data_dir, "train_data.npz"))["X"]).float()
    y_train = torch.from_numpy(np.load(os.path.join(data_dir, "train_data.npz"))["y"]).long()
    X_val = torch.from_numpy(np.load(os.path.join(data_dir, "val_data.npz"))["X"]).float()
    y_val = torch.from_numpy(np.load(os.path.join(data_dir, "val_data.npz"))["y"]).long()

    train_loader = DataLoader(TensorDataset(X_train, y_train), batch_size=CONFIG["batch_size"], shuffle=True)
    val_loader = DataLoader(TensorDataset(X_val, y_val), batch_size=CONFIG["batch_size"])

    model = SLRHybridModel(CONFIG["num_classes"], CONFIG["input_dim"], CONFIG["hidden_dim"]).to(device)
    

    optimizer = torch.optim.Adam(model.parameters(), lr=CONFIG["lr"])
    criterion = nn.CrossEntropyLoss(label_smoothing=0.1)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=5)
    
    mlflow.set_tracking_uri(CONFIG["mlflow_tracking_uri"])
    mlflow.set_experiment(CONFIG["experiment_name"])
    with mlflow.start_run():
        mlflow.log_params(CONFIG)
        mlflow.log_param("param_count", sum(p.numel() for p in model.parameters()))

        best_val_loss = float('inf')
        early_stop_counter = 0

        for epoch in range(CONFIG["epochs"]):
            model.train()
            train_loss, correct = 0, 0
            for X, y in train_loader:
                X, y = X.to(device), y.to(device)
                optimizer.zero_grad()
                output = model(X)
                loss = criterion(output, y)
                loss.backward()
                optimizer.step()
                
                train_loss += loss.item()
                correct += (output.argmax(1) == y).type(torch.float).sum().item()


            model.eval()
            val_loss, val_correct = 0, 0
            with torch.no_grad():
                for X, y in val_loader:
                    X, y = X.to(device), y.to(device)
                    output = model(X)
                    val_loss += criterion(output, y).item()
                    val_correct += (output.argmax(1) == y).type(torch.float).sum().item()


            metrics = {
                "train_loss": train_loss / len(train_loader),
                "train_acc": correct / len(X_train),
                "val_loss": val_loss / len(val_loader),
                "val_acc": val_correct / len(X_val)
            }
            mlflow.log_metrics(metrics, step=epoch)
            scheduler.step(metrics["val_loss"])

            print(f"Epoch {epoch}: Train Loss {metrics['train_loss']:.4f}, Train Acc {metrics['train_acc']:.4f}, Val Loss {metrics['val_loss']:.4f}, Val Acc {metrics['val_acc']:.4f}")


            if metrics["val_loss"] < best_val_loss:
                best_val_loss = metrics["val_loss"]
                torch.save(model.state_dict(), "best_hybrid_model.pt")
                early_stop_counter = 0
            else:
                early_stop_counter += 1
            
            if early_stop_counter >= CONFIG["patience"]:
                print("Early stopping triggered.")
                break

        mlflow.pytorch.log_model(model, "model")

if __name__ == "__main__":
    train_hybrid()

2026/04/01 23:47:21 INFO mlflow.store.db.utils: Creating initial MLflow database tables...
2026/04/01 23:47:21 INFO mlflow.store.db.utils: Updating database tables
2026/04/01 23:47:22 INFO mlflow.tracking.fluent: Experiment with name 'SSL400-Hybrid-Paper-Replication' does not exist. Creating a new experiment.


Epoch 0: Train Loss 5.7649, Train Acc 0.0172, Val Loss 5.3351, Val Acc 0.0389
Epoch 1: Train Loss 5.3942, Train Acc 0.0309, Val Loss 4.9659, Val Acc 0.0467
Epoch 2: Train Loss 5.1951, Train Acc 0.0381, Val Loss 4.7832, Val Acc 0.0856
Epoch 3: Train Loss 5.0661, Train Acc 0.0528, Val Loss 4.6082, Val Acc 0.0973
Epoch 4: Train Loss 4.9652, Train Acc 0.0550, Val Loss 4.5223, Val Acc 0.1128
Epoch 5: Train Loss 4.8384, Train Acc 0.0665, Val Loss 4.4096, Val Acc 0.1128
Epoch 6: Train Loss 4.7579, Train Acc 0.0715, Val Loss 4.3188, Val Acc 0.1479
Epoch 7: Train Loss 4.6562, Train Acc 0.0765, Val Loss 4.2988, Val Acc 0.1440
Epoch 8: Train Loss 4.5735, Train Acc 0.0884, Val Loss 4.2170, Val Acc 0.1440
Epoch 9: Train Loss 4.4931, Train Acc 0.1042, Val Loss 4.1515, Val Acc 0.1673
Epoch 10: Train Loss 4.4309, Train Acc 0.0999, Val Loss 4.0840, Val Acc 0.1595
Epoch 11: Train Loss 4.3622, Train Acc 0.1092, Val Loss 4.0140, Val Acc 0.1751
Epoch 12: Train Loss 4.2963, Train Acc 0.1229, Val Loss 3.9610

2026/04/01 23:49:42 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/04/01 23:49:42 WARNING mlflow.pytorch: Saving pytorch model by Pickle or CloudPickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization.The recommended safe alternative is to set 'export_model' to True to save the pytorch model using the safe graph model format.


Epoch 99: Train Loss 2.2993, Train Acc 0.6730, Val Loss 2.9578, Val Acc 0.5292
